In [4]:
from xml.parsers import expat

def branch_text_positions(xml_string):
    
    parser = expat.ParserCreate()

    results = []
    stack = []

    text_pos = 0  # position in tag-stripped text

    def to_branch( name ):
        tag_names = [ item['tag'] for item in stack ]
        tag_names.append( name )
        return '.'.join( tag_names )

    def start(name, attrs):
        nonlocal text_pos
        stack.append({
            "tag": name,
            "start": text_pos
        })

    def end(name):
        nonlocal text_pos
        item = stack.pop()
        item['tag'] = to_branch( item['tag'] )
        item['end'] = text_pos
        results.append(item)

    def char_data(data):
        nonlocal text_pos
        text_pos += len(data)

    parser.StartElementHandler = start
    parser.EndElementHandler = end
    parser.CharacterDataHandler = char_data

    parser.Parse(xml_string)

    return results

In [ ]:
test_tei = '''
<tei>
	<text>
		<l>some<note><text><l>a note</l></text></note>text</l>
		<l>some<note><text><l>a note</l></text></note>text</l>
		<l>some<div><text>a div</text></div>text</l>
    </text>
	<text>
		<l>some<note><text><l>a note</l></text></note>text</l>
    </text>
</tei>
'''

list( reversed( branch_text_positions( test_tei ) ) )


[{'tag': 'tei', 'start': 0, 'end': 82},
 {'tag': 'tei.text', 'start': 59, 'end': 81},
 {'tag': 'tei.text.l', 'start': 62, 'end': 76},
 {'tag': 'tei.text.l.note', 'start': 66, 'end': 72},
 {'tag': 'tei.text.l.note.text', 'start': 66, 'end': 72},
 {'tag': 'tei.text.l.note.text.l', 'start': 66, 'end': 72},
 {'tag': 'tei.text', 'start': 2, 'end': 57},
 {'tag': 'tei.text.l', 'start': 39, 'end': 52},
 {'tag': 'tei.text.l.div', 'start': 43, 'end': 48},
 {'tag': 'tei.text.l.div.text', 'start': 43, 'end': 48},
 {'tag': 'tei.text.l', 'start': 22, 'end': 36},
 {'tag': 'tei.text.l.note', 'start': 26, 'end': 32},
 {'tag': 'tei.text.l.note.text', 'start': 26, 'end': 32},
 {'tag': 'tei.text.l.note.text.l', 'start': 26, 'end': 32},
 {'tag': 'tei.text.l', 'start': 5, 'end': 19},
 {'tag': 'tei.text.l.note', 'start': 9, 'end': 15},
 {'tag': 'tei.text.l.note.text', 'start': 9, 'end': 15},
 {'tag': 'tei.text.l.note.text.l', 'start': 9, 'end': 15}]

In [30]:
for xml_path in [ item['tag'] for item in list( reversed( branch_text_positions( test_tei ) ) ) ]:
    print( xml_path )

tei
tei.text
tei.text.l
tei.text.l.note
tei.text.l.note.text
tei.text.l.note.text.l
tei.text
tei.text.l
tei.text.l.div
tei.text.l.div.text
tei.text.l
tei.text.l.note
tei.text.l.note.text
tei.text.l.note.text.l
tei.text.l
tei.text.l.note
tei.text.l.note.text
tei.text.l.note.text.l


In [2]:
from glob import glob
import pandas
from random import randrange

bp_dfs = {}

file_paths = glob( './data/edition_level/*.xml' )

print( len( file_paths ) )

15


In [5]:
file_count = 0

for file_path in file_paths:
    with open( file_path, 'r' ) as xml_file:
        try:
            xml = xml_file.read()
        except Exception as e:
            print( '')
            print( str( e ), file_path )
            with open( file_path, 'r', encoding='latin-1' ) as xml_file:
                xml = xml_file.read()
        bp = branch_text_positions( xml )
        bp_df = pandas.DataFrame( bp )
        bp_dfs[ file_path ] = bp_df
    file_count += 1
    print( file_count, end='' )
    if file_count % 30 == 0:
        print( ' ' )
    else:
        print( ', ', end='' )
        
# Quick check…
result_from_one_file = list( bp_dfs.keys() )[ randrange( len( file_paths ) ) ]
print( '\n\n', result_from_one_file, '=>' )
bp_dfs[ result_from_one_file ]

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 

 ./data/edition_level/Digital-marco-polo.xml =>


,tag,start,end
0,TEI.teiHeader.fileDesc.titleStmt.title,39,46
1,TEI.teiHeader.fileDesc.titleStmt.respStmt.resp,84,93
2,TEI.teiHeader.fileDesc.titleStmt.respStmt.name,114,128
3,TEI.teiHeader.fileDesc.titleStmt.respStmt,63,145
4,TEI.teiHeader.fileDesc.titleStmt,22,158
...,...,...,...
11172,TEI.text.body.div.list,405088,409036
11173,TEI.text.body.div,374497,409048
11174,TEI.text.body,374487,409055
11175,TEI.text,374480,409059


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas

doc_vectorizer = TfidfVectorizer(
    token_pattern = None,
    tokenizer = lambda s: s.split( ' ' ),  # split on space
    lowercase = False                      # keep as-is
)

branch_docs = []

for key in bp_dfs.keys():
    bp_df = bp_dfs[key]
    print( f'{len( bp_df['tag'].unique() )} unique branches in {key}' )
    branch_doc = ' '.join( bp_df['tag'].values )
    branch_docs.append( branch_doc )

tfidf_docs = doc_vectorizer.fit_transform( branch_docs )

104 unique branches in ./data/edition_level/Digital-marco-polo.xml
265 unique branches in ./data/edition_level/Chartes-de-labbaye-de-Saint-Denis.xml
186 unique branches in ./data/edition_level/Petrarchive.xml
92 unique branches in ./data/edition_level/leopardi(_).xml
541 unique branches in ./data/edition_level/beckettarchive.xml
47 unique branches in ./data/edition_level/Chronik-Ulrich-Richental.xml
230 unique branches in ./data/edition_level/Der arme Heinrich.xml
120 unique branches in ./data/edition_level/Lyrik-des-deutschen-Mittelalters.xml
93 unique branches in ./data/edition_level/4earlyChanTextsFromDunhuang_aTEIbasedEdition2017.xml
14 unique branches in ./data/edition_level/CURSUS-OnlineResourceOfMedievalLiturgicalTexts.xml
92 unique branches in ./data/edition_level/Van-gogh.xml
791 unique branches in ./data/edition_level/CoReMa.xml
121 unique branches in ./data/edition_level/Faust.xml
155 unique branches in ./data/edition_level/Kaiserchronik.xml
258 unique branches in ./data/edi

In [8]:
branch_doc

'TEI.teiHeader.fileDesc.titleStmt.title TEI.teiHeader.fileDesc.titleStmt.title TEI.teiHeader.fileDesc.titleStmt.author TEI.teiHeader.fileDesc.titleStmt.respStmt.resp TEI.teiHeader.fileDesc.titleStmt.respStmt.name TEI.teiHeader.fileDesc.titleStmt.respStmt.name TEI.teiHeader.fileDesc.titleStmt.respStmt TEI.teiHeader.fileDesc.titleStmt.respStmt.resp TEI.teiHeader.fileDesc.titleStmt.respStmt.name TEI.teiHeader.fileDesc.titleStmt.respStmt TEI.teiHeader.fileDesc.titleStmt TEI.teiHeader.fileDesc.editionStmt.edition.interpGrp.interp TEI.teiHeader.fileDesc.editionStmt.edition.interpGrp TEI.teiHeader.fileDesc.editionStmt.edition TEI.teiHeader.fileDesc.editionStmt TEI.teiHeader.fileDesc.publicationStmt.authority TEI.teiHeader.fileDesc.publicationStmt TEI.teiHeader.fileDesc.sourceDesc.listWit.witness.msDesc.msIdentifier.settlement TEI.teiHeader.fileDesc.sourceDesc.listWit.witness.msDesc.msIdentifier.repository TEI.teiHeader.fileDesc.sourceDesc.listWit.witness.msDesc.msIdentifier.idno TEI.teiHeader

In [9]:
tfidf_docs.shape

(15, 2720)

In [11]:
from sklearn.cluster import HDBSCAN
labels = HDBSCAN( min_cluster_size=2, metric='cosine' ).fit_predict( tfidf_docs )

/Users/joris/Workspace/THINK/conflint/representation/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


In [12]:
labels

array([ 1, -1,  3,  3,  2,  1,  0,  0, -1, -1, -1, -1,  2,  2, -1])

In [15]:
import numpy as np

In [17]:
feature_names = np.array(doc_vectorizer.get_feature_names_out())

tfidf_docs_df = pandas.DataFrame( tfidf_docs.toarray() )
tfidf_docs_df['cluster'] = labels

cluster_editions = pandas.DataFrame( [ file_path.split('/')[-1].split('.')[-2] for file_path in file_paths ], index=labels )

print( feature_names )
print( 'Number of features:', len( feature_names ) )
display( cluster_editions )
tfidf_docs_df

['CURSUS' 'CURSUS.text' 'CURSUS.text.body' ...
 'div.text.body.div.ab.title.space' 'div.text.body.div.ab.unclear'
 'div.text.body.div.ab.unclear.pc']
Number of features: 2720


,0
1,Digital-marco-polo
-1,Chartes-de-labbaye-de-Saint-Denis
3,Petrarchive
3,leopardi(_)
2,beckettarchive
1,Chronik-Ulrich-Richental
0,Der arme Heinrich
0,Lyrik-des-deutschen-Mittelalters
-1,4earlyChanTextsFromDunhuang_aTEIbasedEdition2017
-1,CURSUS-OnlineResourceOfMedievalLiturgicalTexts


,0,1,2,3,4,5,6,7,8,9,...,2711,2712,2713,2714,2715,2716,2717,2718,2719,cluster
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,1
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,-1
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,3
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,3
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,2
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,1
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,0
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.003907,0.003447,0.00023,0.002298,0.001839,0.000689,0.00023,0.007125,0.00046,-1
9,0.003853,0.003853,0.003853,0.030822,0.446925,0.446925,0.446925,0.446925,0.446925,0.007706,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.00000,-1


In [34]:
# Mean TF-IDF weight per term per cluster
cluster_means = tfidf_docs_df.groupby('cluster').mean()

# Top terms per cluster
report = ''
for idx, c in enumerate( cluster_means.index ):
    top = cluster_means.loc[c].sort_values(ascending=False).head(15)
    report += f'\n--- Cluster {idx} --- \n\nEditions:\n'
    for edition in cluster_editions.loc[c][0]:
        report += f'  * {edition}\n'
    report += '\nTerms frequently used in this cluster, rare in others:\n'
    for item in ( list( zip( feature_names[top.index.astype(int)], top[:15] ) ) ):
        report += f'  ({round( item[1], 4 ):.4f})  {item[0]}\n'
    report += f'{{sub_report_{idx}}}\n'

print( report )


--- Cluster 0 --- 

Editions:
  * Chartes-de-labbaye-de-Saint-Denis
  * 4earlyChanTextsFromDunhuang_aTEIbasedEdition2017
  * CURSUS-OnlineResourceOfMedievalLiturgicalTexts
  * Van-gogh
  * CoReMa
  * saba

Terms frequently used in this cluster, rare in others:
  (0.1442)  TEI.text.body.ab.seg.g
  (0.1369)  TEI.text.body.div.ab.lb
  (0.1244)  TEI.text.body.div.div.div.div.lg.l
  (0.1171)  div.text.body.div.ab.orig
  (0.0882)  div.text.body.div.ab.pc
  (0.0835)  TEI.text.group.text.body.div.listWit.listWit.witness
  (0.0820)  TEI.text.body.div.div.div.div.div.lg.l
  (0.0767)  div.text.body.div.ab.orig.g
  (0.0745)  CURSUS.text.body.chapter.verse.reference
  (0.0745)  CURSUS.text.body.chapter.verse.reference.bookName
  (0.0745)  CURSUS.text.body.chapter.verse.l
  (0.0745)  CURSUS.text.body.chapter.verse
  (0.0745)  CURSUS.text.body.chapter.verse.reference.verseNum
  (0.0502)  TEI.text.body.div.ab.c
  (0.0480)  TEI.text.body.div.ab
{sub_report_0}

--- Cluster 1 --- 

Editions:
  * Der arm

The above tells us what features are most used in the documents pertaining to a cluster.

We can use some different logic to see what features actually discriminate between clusters.

In [35]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, C=1.0 )
clf.fit(tfidf_docs, labels)

sub_reports = []

for i, c in enumerate(clf.classes_):
    coefs = clf.coef_[i]
    top = np.argsort(coefs)[-15:][::-1]
    sub_report = f"\nCluster {i} Top distinguishing terms:\n"
    for item in list( zip( feature_names[top], coefs[top].round(3) ) ):
        sub_report += f'  ({round( item[1], 4 ):.4f})  {item[0]}\n'
    sub_reports.append( sub_report )

report = report.format_map( {f'sub_report_{i}': s for i, s in enumerate( sub_reports ) } )

print( report )

with open( './distinguishing_tei_editions_simpler.md', 'w' ) as report_file:
    report_file.write( report )


--- Cluster 0 --- 

Editions:
  * Chartes-de-labbaye-de-Saint-Denis
  * 4earlyChanTextsFromDunhuang_aTEIbasedEdition2017
  * CURSUS-OnlineResourceOfMedievalLiturgicalTexts
  * Van-gogh
  * CoReMa
  * saba

Terms frequently used in this cluster, rare in others:
  (0.1442)  TEI.text.body.ab.seg.g
  (0.1369)  TEI.text.body.div.ab.lb
  (0.1244)  TEI.text.body.div.div.div.div.lg.l
  (0.1171)  div.text.body.div.ab.orig
  (0.0882)  div.text.body.div.ab.pc
  (0.0835)  TEI.text.group.text.body.div.listWit.listWit.witness
  (0.0820)  TEI.text.body.div.div.div.div.div.lg.l
  (0.0767)  div.text.body.div.ab.orig.g
  (0.0745)  CURSUS.text.body.chapter.verse.reference
  (0.0745)  CURSUS.text.body.chapter.verse.reference.bookName
  (0.0745)  CURSUS.text.body.chapter.verse.l
  (0.0745)  CURSUS.text.body.chapter.verse
  (0.0745)  CURSUS.text.body.chapter.verse.reference.verseNum
  (0.0502)  TEI.text.body.div.ab.c
  (0.0480)  TEI.text.body.div.ab

Cluster 0 Top distinguishing terms:
  (0.3910)  TEI.text